# Wav2Lip Lip-Sync on Colab **T4**

Makes a face (video or image) speak an audio clip.

### Runs on the Colab T4 only — NOT HuggingFace ZeroGPU.
Clones open-source Wav2Lip and runs `inference.py` on the T4 Colab gives your
runtime. It never calls the hosted HF Space, so ZeroGPU is never involved.

**First:** Runtime -> Change runtime type -> **T4 GPU** -> Save. Then run the
cells top to bottom. Step 0 proves you got a T4.

### The one setting that matters: `OUT_HEIGHT` (Step 4).
Wav2Lip resizes every frame to `OUT_HEIGHT` pixels tall *before* processing.
Its default (480) is TINY — it shrinks a distant face until detection fails and
the output is blurry. For a wide shot with a small/distant face, set
`OUT_HEIGHT` HIGH (1408, or 2112 if the face is very small). That makes the
face big enough to auto-detect AND keeps the output sharp. The full frame (the
whole parking lot) is always preserved — Wav2Lip only replaces the mouth.


## Step 0 - Prove the GPU is a T4 (not ZeroGPU, not CPU)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then rerun.'
print('Torch is using:', torch.cuda.get_device_name(0), '- Colab hardware, not HF ZeroGPU.')


## Step 1 - Get Wav2Lip (maintained fork) and install deps


In [ ]:
import os
if not os.path.isdir('/content/Wav2Lip'):
    !git clone -q https://github.com/justinjohn0306/Wav2Lip /content/Wav2Lip
%cd /content/Wav2Lip
!pip install -q -r requirements.txt
!pip install -q batch-face
print('deps installed')


## Step 2 - Download the model checkpoints


In [ ]:
%cd /content/Wav2Lip
import os
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('face_detection/detection/sfd', exist_ok=True)
B='https://github.com/justinjohn0306/Wav2Lip/releases/download/models/'
!wget -q -c $B'wav2lip_gan.pth' -O checkpoints/wav2lip_gan.pth
!wget -q -c $B's3fd.pth'        -O face_detection/detection/sfd/s3fd.pth
print('checkpoints:', os.listdir('checkpoints'))


## Step 3 - Upload your inputs
**FACE** = your clip (e.g. the WIDE `joker-walk-14b` clip — this is fine, you
keep the whole shot). **AUDIO** = the voice line (wav/mp3).


In [ ]:
from google.colab import files
print('Upload the FACE (mp4 / png / jpg):')
FACE = '/content/Wav2Lip/' + list(files.upload().keys())[0]
print('Upload the AUDIO (wav / mp3):')
AUDIO = '/content/Wav2Lip/' + list(files.upload().keys())[0]
print('FACE :', FACE)
print('AUDIO:', AUDIO)


## Step 4 - Run Wav2Lip on the T4
- **OUT_HEIGHT** is the key knob. Start at **1408**. If you get
  `Face not detected!`, raise it to **2112**. Higher = bigger face = better
  detection + sharper output (a bit slower).
- **PADS** `[top,bottom,left,right]`: grow the mouth box; raise bottom if the
  chin is clipped.
- **BOX** `[top,bottom,left,right]`: only if detection still fails at
  OUT_HEIGHT 2112 — coordinates are on the OUT_HEIGHT-tall frame.
- Leave **CROP = None** for a wide shot (CROP discards the rest of the frame).

This prints the full error if inference fails, so you can see the real cause.


In [ ]:
OUT_HEIGHT = 1408      # start here; raise to 2112 if 'Face not detected'
PADS = [0, 10, 0, 0]   # top bottom left right
BOX  = None            # last resort: [top,bottom,left,right] on the OUT_HEIGHT frame
CROP = None            # leave None for a wide shot

cmd = ['python','inference.py',
       '--checkpoint_path','checkpoints/wav2lip_gan.pth',
       '--face', FACE, '--audio', AUDIO,
       '--outfile','/content/result.mp4',
       '--out_height', str(OUT_HEIGHT),
       '--nosmooth','--pads', *map(str, PADS)]
if BOX:  cmd += ['--box',  *map(str, BOX)]
if CROP: cmd += ['--crop', *map(str, CROP)]
print('running:', ' '.join(cmd))
import subprocess
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-3000:])
if p.returncode != 0:
    print('--- ERROR ---'); print(p.stderr[-3000:])
else:
    print('done -> /content/result.mp4')


## Step 5 - (optional) resize the result to a target width
The result is at OUT_HEIGHT resolution. Set a `TARGET_W` to standardize it
(e.g. 1280 to match a Wan clip), or set `TARGET_W = 0` to keep it as-is.


In [ ]:
TARGET_W = 1280   # 0 = keep OUT_HEIGHT resolution
import os
FINAL = '/content/result.mp4'
if TARGET_W:
    FINAL = '/content/result_final.mp4'
    os.system(f'ffmpeg -y -loglevel error -i /content/result.mp4 -vf "scale={TARGET_W}:-2:flags=lanczos" -c:a aac "{FINAL}"')
    print('resized to width', TARGET_W, '->', FINAL)
else:
    print('kept as-is ->', FINAL)


## Step 6 - Preview and download


In [ ]:
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open(FINAL,'rb').read()).decode()
HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')


In [ ]:
from google.colab import files
files.download(FINAL)  # saves to your Downloads; move it to D:\\MatrixVideos


---
### Still low-res or undetected?
1. Raise `OUT_HEIGHT` to 2112 (Step 4).
2. The mouth sharpness is ultimately capped by how many real pixels the face
   has in your SOURCE clip. A distant figure has little face detail, so the
   synced mouth stays soft even at high OUT_HEIGHT - that is expected at
   wide-shot scale. For a crisp talking face, the character needs to be closer
   to camera in the shot.
3. This fork has no built-in face enhancer; a separate GFPGAN pass could
   sharpen the face further (ask and it can be added).
